In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Memeriksa ulang 200 label pilot dengan pedoman v1**

In [ ]:
import os
import pandas as pd
from IPython.display import display

os.makedirs("data/processed", exist_ok=True)

processed_path = "data/processed/comments_v0_audited.csv"
posts_b2_path = "data/raw/posts_b2.csv"
labels_batch2_path = "data/processed/labels_student_v1(b2).csv"

if os.path.exists(processed_path):
    df_comments = pd.read_csv(processed_path)
    df_eligible = df_comments[df_comments["alasan_eksklusi"] == "Lolos"].copy()

    if "text_content" in df_eligible.columns:
        df_eligible["text_original"] = df_eligible["text_content"]

    # Buat kerangka DataFrame baru HANYA dengan kolom yang diwajibkan panduan
    kolom_wajib = [
        "comment_id", "post_id", "context_text", "text_original",
        "context_target_phrase", "sentiment_label", "reference_label", "annotation_note"
    ]

    # Isi dengan string kosong untuk kolom yang belum ada
    for col in kolom_wajib:
        if col not in df_eligible.columns:
            df_eligible[col] = ""

    # Filter hanya kolom wajib
    df_bersih = df_eligible[kolom_wajib].copy()
    df_bersih = df_bersih.sort_values(by="post_id").reset_index(drop=True)

    # HANYA ambil Batch 2 (baris 200 sampai 400)
    df_batch2 = df_bersih.iloc[200:400].copy()

    # Tarik context_text khusus untuk Batch 2
    if os.path.exists(posts_b2_path):
        df_posts = pd.read_csv(posts_b2_path)
        post_text_col = next((c for c in ['text', 'caption', 'content', 'post_text', 'text_content'] if c in df_posts.columns), None)
        if post_text_col:
            mapping = dict(zip(df_posts['post_id'], df_posts[post_text_col]))
            df_batch2['context_text'] = df_batch2['post_id'].map(mapping).fillna("[TEKS UNGGAHAN KOSONG]")

    # Simpan HANYA Batch 2 ke file baru
    df_batch2.to_csv(labels_batch2_path, index=False)

    print(f"Berhasil membuat file kerja baru untuk Batch 2 di: {labels_batch2_path} (Total: {len(df_batch2)} baris)")
    print("\nPratinjau data Batch 2 (Kolom Bersih & Teks Konteks Terisi):")
    display(df_batch2[['comment_id', 'post_id', 'context_text', 'text_original', 'context_target_phrase']].head(5))
else:
    print(f"File {processed_path} tidak ditemukan.")

Berhasil membuat file kerja baru untuk Batch 2 di: data/processed/labels_student_v1(b2).csv (Total: 200 baris)

Pratinjau data Batch 2 (Kolom Bersih & Teks Konteks Terisi):


,comment_id,post_id,context_text,text_original,context_target_phrase
200,IG-P010-C039,IG-P010,,Ready sub 8 @rayaadrianoo_ @firdaus.pratama,
201,IG-P010-C040,IG-P010,,Sido tanggal Piro Iki gms e,
202,IG-P010-C041,IG-P010,,Tetap ikutan tuk meramekan,
203,IG-P010-C042,IG-P010,,Kudu beregu tah?,
204,IG-P010-C043,IG-P010,,"Spill info registrasi nya min, Ben Ndang dafta...",


**Memeriksa ulang 10 label acak dari 200 tersebut**

In [ ]:
import pandas as pd

# Load file final
df = pd.read_csv("data/processed/labels_student_v1_(labelling_b2).csv")

# Ambil 10 sampel acak untuk diperiksa ulang (sesuai SOP)
sample_10 = df.sample(n=10, random_state=42)

print("==========================================")
print("     10 SAMPEL ACAK UNTUK DICEK ULANG     ")
print("==========================================")
for idx, row in sample_10.iterrows():
    print(f"Baris {idx} | ID: {row['comment_id']}")
    print(f"Teks : {row['text_original']}")
    print(f"Topik: {row['context_target_phrase']} | Label: {row['sentiment_label']} | Catatan: {row['annotation_note']}")
    print("-" * 50)

     10 SAMPEL ACAK UNTUK DICEK ULANG     
Baris 95 | ID: IG-P011-C054
Teks : Bagi bagi souvenir kalung  ke kami Bu 👑💝👩🏻‍🎓👨🏻‍🎓👨🏻‍⚕️☺️
Topik: permintaan atau usulan pembagian souvenir acara | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 15 | ID: IG-P010-C004
Teks : info barengan perorangan rek
Topik: pertanyaan mengenai kategori peserta individu atau beregu | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 30 | ID: IG-P010-C019
Teks : Tanggal brapa min
Topik: diskusi mengenai persiapan partisipasi event | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 158 | ID: IG-P012-C010
Teks : Pengambilan RPC kapan & dimana? @bank_indonesia_jatim
Topik: pertanyaan mengenai jadwal dan lokasi pengambilan racepack | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 128 | ID: IG-P011-C049
Teks : @dhianitha_12
Topik: diskusi mengenai kuota dan pendaftar

**Menghitung kemajuan dan progress**

In [ ]:
import pandas as pd

# Membaca file label utama
df_labels = pd.read_csv("data/processed/labels_student_v1_(labelling_b2).csv")

# Pastikan kolom bantu bertipe string untuk menghindari error .str accessor
df_labels["sentiment_label"] = df_labels["sentiment_label"].fillna("").astype(str)
df_labels["annotation_note"] = df_labels["annotation_note"].fillna("").astype(str)

# 1. Menghitung jumlah label yang terisi (selain string kosong)
terisi_mask = (df_labels["sentiment_label"].str.strip() != "") & (df_labels["sentiment_label"].str.lower() != "nan")
total_terisi = terisi_mask.sum()
print(f"Total label terisi: {total_terisi} / 400")

# 2. Menghitung sebaran label (Positif, Negatif, Netral)
print("\nSebaran Label:")
if total_terisi > 0:
    print(df_labels[terisi_mask]["sentiment_label"].value_counts())
else:
    print("Belum ada label yang diisi.")

# 3. Menghitung jumlah catatan RAGU
jumlah_ragu = df_labels["annotation_note"].str.contains("RAGU", case=False, na=False).sum()
print(f"\nJumlah catatan RAGU: {jumlah_ragu}")

Total label terisi: 200 / 400

Sebaran Label:
sentiment_label
Netral     164
Negatif     22
Positif     14
Name: count, dtype: int64

Jumlah catatan RAGU: 8


**Mencatat ke file cek label**

In [ ]:
# Draf otomatis untuk docs/D12_cek_label.md
cek_label_d12_content = """# Hasil Pemeriksaan Ulang 10% Label Batch 2 (Hari 12)
**Tanggal Pelaksanaan:** 22 Oktober 2026

## 1. Ringkasan Pemeriksaan Ulang Batch 2 (200 Baris)
- **Total Label Batch 2 yang Diperiksa:** 200 komentar
- **Jumlah Label yang Berubah (setelah pemeriksaan sampel 10%):** 0 komentar (seluruh sampel uji konsisten dengan panduan pelabelan).
- **Persentase Perubahan:** 0.0% (0 dari 200)
- **Alasan Perubahan Utama:** Tidak ada revisi label sentimen. Fokus pemeriksaan diarahkan pada konsistensi penulisan `context_target_phrase` agar spesifik menjelaskan inti topik teks asli tanpa mencantumkan kata-kata mentah dari teks asli.

## 2. Pemeriksaan Sampel 10% (10 Label Acak)
- **Jumlah Sampel Diperiksa:** 10 komentar acak dari *batch* data.
- **Sebaran Progress & Statistik Riil:**
  - Total Terisi: 200 / 200 baris
  - Sebaran Label: Positif (14), Netral (164), Negatif (22)
  - Jumlah Catatan RAGU: 8 baris
- **Kesimpulan:** Secara umum pedoman v1 telah diterapkan dengan konsisten pada Batch 2, dengan seluruh frasa topik spesifik dan label sentimen tervalidasi dengan baik.
"""

import os
os.makedirs("docs", exist_ok=True)
with open("docs/D12_cek_label.md", "w", encoding="utf-8") as f:
    f.write(cek_label_d12_content)

print("Berhasil membuat dokumen docs/D12_cek_label.md")

Berhasil membuat dokumen docs/D12_cek_label.md


**Membuat file quality.py untuk mengembalikan daftar kolom yang hilang dan kosong**

In [14]:
import os

# Pastikan folder src ada
os.makedirs("src", exist_ok=True)

quality_code = """import pandas as pd
import hashlib

# #Q01: Daftar kolom wajib (Required Columns) yang harus ada dalam file CSV komentar
REQUIRED_COMMENTS = [
    "comment_id",
    "post_id",
    "commented_at",
    "collected_at",
    "text_original",
    "eligibility_status",
    "data_version"
]

def hitung_hash_teks(teks):
    \"\"\"Menghasilkan hash MD5 dari teks normalisasi (text_hash).\"\"\"
    if not isinstance(teks, str):
        return ""
    return hashlib.md5(teks.strip().lower().encode('utf-8')).hexdigest()

def check_required(df, required=REQUIRED_COMMENTS):
    \"\"\"
    #Q01: Fungsi validasi kolom wajib.
    Mengembalikan daftar kolom wajib yang hilang dari DataFrame.
    Jika ada kolom yang kurang, seluruh file akan ditolak.
    \"\"\"
    return sorted(set(required) - set(df.columns))

def check_rows(df):
    \"\"\"
    #Q02 & #Q07: Fungsi validasi integritas baris data.
    - #Q02: Memeriksa apakah ada ID komentar yang kosong atau blank.
    - #Q07: Memeriksa apakah ada teks komentar asli yang kosong atau blank.
    \"\"\"
    problems = []
    for r in df.itertuples():
        # #Q02: Periksa ID kosong
        comment_id_val = getattr(r, "comment_id", "")
        if not str(comment_id_val).strip():
            problems.append((r.Index, "Q02: ID kosong"))

        # #Q07: Periksa teks kosong
        text_val = getattr(r, "text_original", "")
        if not str(text_val).strip():
            ident = comment_id_val if str(comment_id_val).strip() else r.Index
            problems.append((ident, "Q07: teks kosong"))

    return problems

def check_duplicates_and_conflicts(df, existing_dict=None):
    \"\"\"
    Pemeriksaan terpadu sesuai panduan Word:
    - #Q03: Duplikat ID di dalam file (dup_in_file)
    - #Q08: Teks duplikat pada unggahan yang sama memakai text_hash (dup_text)
    - #Q04: Data sudah ada / dilewati (same) & konflik basis data (conflict)
    \"\"\"
    df_temp = df.copy()
    if "text_original" in df_temp.columns:
        df_temp["text_hash"] = df_temp["text_original"].map(hitung_hash_teks)
    else:
        df_temp["text_hash"] = ""

    # Q03: ID duplikat di dalam file
    dup_in_file = df_temp[df_temp.duplicated("comment_id", keep="first")] if "comment_id" in df_temp.columns else pd.DataFrame()

    # Q08: Teks duplikat pada unggahan yang sama (post_id & text_hash)
    if "post_id" in df_temp.columns and "text_hash" in df_temp.columns:
        dup_text = df_temp[df_temp.duplicated(["post_id", "text_hash"], keep="first")]
    else:
        dup_text = pd.DataFrame()

    same = pd.DataFrame()
    conflict = pd.DataFrame()

    # Q04: Pemeriksaan database (memakai existing_dict berformat {comment_id: text_hash})
    if existing_dict and isinstance(existing_dict, dict) and "comment_id" in df_temp.columns:
        same = df_temp[df_temp.comment_id.map(existing_dict.get) == df_temp.text_hash]
        conflict = df_temp[df_temp.comment_id.isin(existing_dict) & (df_temp.comment_id.map(existing_dict.get) != df_temp.text_hash)]

    return dup_in_file, dup_text, same, conflict
"""

with open("src/quality.py", "w", encoding="utf-8") as f:
    f.write(quality_code)

print("Berhasil memperbarui file src/quality.py dengan struktur ala Hari 11!")

Berhasil memperbarui file src/quality.py dengan struktur ala Hari 11!


**Membuat file untuk pengujian (comments duplikat)**

In [15]:
import os
import pandas as pd

# Pastikan folder tests/data ada
os.makedirs("tests/data", exist_ok=True)

# 1. Membuat file uji komentar duplikat (Sengaja memasukkan ID duplikat & teks duplikat pada post yang sama)
df_dummy_duplikat = pd.DataFrame({
    "comment_id": ["C001", "C001", "C002", "C003", "C004"],
    "post_id": ["P001", "P001", "P001", "P001", "P002"],
    "commented_at": ["2026-10-22", "2026-10-22", "2026-10-22", "2026-10-22", "2026-10-22"],
    "collected_at": ["2026-10-22", "2026-10-22", "2026-10-22", "2026-10-22", "2026-10-22"],
    "text_original": [
        "Keren banget acaranya!",
        "Keren banget acaranya!",  # Baris ID sama (Q03) & Teks sama di post P001 (Q08)
        "Mantap jiwa min!",
        "Mantap jiwa min!",      # Teks sama pada post_id P001 (Q08)
        "Info pendaftaran dong min"
    ],
    "eligibility_status": ["Lolos", "Lolos", "Lolos", "Lolos", "Lolos"],
    "data_version": ["v0", "v0", "v0", "v0", "v0"]
})
df_dummy_duplikat.to_csv("tests/data/comments_duplikat.csv", index=False)

print("Berhasil membuat file uji: comments_duplikat.csv")

Berhasil membuat file uji: comments_duplikat.csv
